
# 04. Modelado predictivo de IT

**Objetivo:** realizar la primera fase de modelado del TFM para comparar diferentes algoritmos de aprendizaje automático y establecer una primera criba de modelos candidatos.

La unidad de análisis es **empleado-semana**. Se emplean como predictores las 36 variables definidas en la etapa de preparación y como objetivos las cuatro variables de IT futura:

- `IT_Futura_W1`
- `IT_Futura_W2`
- `IT_Futura_W3`
- `IT_Futura_W4`

> **Importante:** este notebook está planteado inicialmente como fase de análisis y comparación. La optimización intensiva de hiperparámetros y la evaluación final sobre el conjunto de test se realizarán después de identificar los modelos más prometedores.


## 1. Configuración y carga de datos

In [10]:

from pathlib import Path
import warnings
import time
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Ruta de trabajo: el notebook y el CSV deben estar en la misma carpeta
BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "Tabla_Entrenamiento_IT_Modelado.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró {DATA_FILE}. "
        "Coloca el notebook en la carpeta donde está el CSV o modifica DATA_FILE."
    )

df = pd.read_csv(DATA_FILE, sep=";", low_memory=False)

print(f"Filas: {len(df):,}")
print(f"Columnas: {df.shape[1]}")
print(f"Periodo: {df['AnioISO'].min()}-{df['SemanaISO'].min():02d} "
      f"a {df['AnioISO'].max()}-{df['SemanaISO'].max():02d}")


Filas: 1,755,108
Columnas: 40
Periodo: 2024-01 a 2026-52


## 2. Definición de predictores y objetivos

In [11]:

TARGETS = [
    "IT_Futura_W1",
    "IT_Futura_W2",
    "IT_Futura_W3",
    "IT_Futura_W4"
]

PREDICTORS = [
    "AnioISO", "SemanaISO", "ID_Semana",
    "ClaseContrato", "CantidadPeriodosContrato", "DiasTrabajadosHastaCorte",
    "FamiliaFuncion", "Centro", "UnidadOrganizativa", "Funcion", "Area", "Servicio",
    "Sexo", "Edad", "TipoJornada", "PorcHorTbjo", "HSem", "HTbjoAnio",
    "Previous1WeekIT", "Previous1WeekITShortTerm", "Previous1WeekITLongTerm",
    "Previous2WeeksIT", "Previous2WeeksITShortTerm", "Previous2WeeksITLongTerm",
    "Previous1MonthIT", "Previous1MonthITShortTerm", "Previous1MonthITLongTerm",
    "Previous3MonthsIT", "Previous3MonthsITShortTerm", "Previous3MonthsITLongTerm",
    "Previous6MonthsIT", "Previous6MonthsITShortTerm", "Previous6MonthsITLongTerm",
    "Previous12MonthsIT", "Previous12MonthsITShortTerm", "Previous12MonthsITLongTerm"
]

assert len(PREDICTORS) == 36
assert all(c in df.columns for c in PREDICTORS + TARGETS)

X = df[PREDICTORS].copy()
y = df[TARGETS].copy()

print(f"Predictores: {len(PREDICTORS)}")
print(f"Objetivos: {len(TARGETS)}")
print("Nulos en objetivos:")
print(y.isna().sum())


Predictores: 36
Objetivos: 4
Nulos en objetivos:
IT_Futura_W1    0
IT_Futura_W2    0
IT_Futura_W3    0
IT_Futura_W4    0
dtype: int64



## 3. Exploración inicial de los objetivos

Antes de entrenar los modelos se revisa la distribución de las cuatro variables objetivo. Esta comprobación es especialmente relevante porque los objetivos presentan una elevada concentración de valores cero.


In [12]:

print("Distribución de los objetivos")
display(y.describe().T)

resumen_targets = pd.DataFrame({
    "Media": y.mean(),
    "Mediana": y.median(),
    "Máximo": y.max(),
    "% ceros": (y.eq(0).mean() * 100).round(2),
    "Nulos": y.isna().sum()
})

display(resumen_targets)


Distribución de los objetivos


,count,mean,std,min,25%,50%,75%,max
IT_Futura_W1,1755108.0,0.284645,1.133606,0.0,0.0,0.0,0.0,5.0
IT_Futura_W2,1755108.0,0.284533,1.133659,0.0,0.0,0.0,0.0,5.0
IT_Futura_W3,1755108.0,0.284400,1.133647,0.0,0.0,0.0,0.0,5.0
IT_Futura_W4,1755108.0,0.284138,1.133403,0.0,0.0,0.0,0.0,5.0


,Media,Mediana,Máximo,% ceros,Nulos
IT_Futura_W1,0.284645,0.0,5,93.68,0
IT_Futura_W2,0.284533,0.0,5,93.69,0
IT_Futura_W3,0.284400,0.0,5,93.70,0
IT_Futura_W4,0.284138,0.0,5,93.71,0


## 4. División temporal Train / Validation / Test

In [13]:

# Se utiliza ID_Semana como eje temporal global.
# No se realiza una división aleatoria para evitar entrenar con información
# temporalmente posterior a la utilizada para evaluar.

weeks = np.sort(df["ID_Semana"].unique())
n_weeks = len(weeks)

# División temporal aproximada 70% / 15% / 15%
train_end = int(n_weeks * 0.70)
val_end = int(n_weeks * 0.85)

train_weeks = weeks[:train_end]
val_weeks = weeks[train_end:val_end]
test_weeks = weeks[val_end:]

train_mask = df["ID_Semana"].isin(train_weeks)
val_mask = df["ID_Semana"].isin(val_weeks)
test_mask = df["ID_Semana"].isin(test_weeks)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print(f"Train: {len(X_train):,} filas | semanas {train_weeks.min()}-{train_weeks.max()}")
print(f"Validation: {len(X_val):,} filas | semanas {val_weeks.min()}-{val_weeks.max()}")
print(f"Test: {len(X_test):,} filas | semanas {test_weeks.min()}-{test_weeks.max()}")


Train: 1,206,215 filas | semanas 1-91
Validation: 273,238 filas | semanas 92-111
Test: 275,655 filas | semanas 112-131


In [5]:

# Comprobación visual/resumen de la separación temporal
split_summary = pd.DataFrame([
    ["Train", train_weeks.min(), train_weeks.max(), len(X_train)],
    ["Validation", val_weeks.min(), val_weeks.max(), len(X_val)],
    ["Test", test_weeks.min(), test_weeks.max(), len(X_test)]
], columns=["Conjunto", "ID_Semana inicial", "ID_Semana final", "Observaciones"])

display(split_summary)


,Conjunto,ID_Semana inicial,ID_Semana final,Observaciones
0,Train,1,91,1206215
1,Validation,92,111,273238
2,Test,112,131,275655



## 5. Preprocesamiento

El tratamiento se integra dentro de `Pipeline` y `ColumnTransformer`, de forma que los parámetros aprendidos durante el entrenamiento no se calculen utilizando Validation o Test.

Las variables categóricas se codifican mediante **One-Hot Encoding**. Las variables numéricas se mantienen sin escalado para los modelos basados en árboles; para el modelo lineal se aplica escalado.

La imputación se mantiene como medida de seguridad dentro del pipeline, aunque la etapa anterior ya realizó el tratamiento de valores ausentes.


In [15]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [c for c in PREDICTORS if c not in categorical_features]

print(f"Variables numéricas: {len(numeric_features)}")
print(f"Variables categóricas: {len(categorical_features)}")
print("\nCategóricas:")
print(categorical_features)


Variables numéricas: 27
Variables categóricas: 9

Categóricas:
['ClaseContrato', 'FamiliaFuncion', 'Centro', 'UnidadOrganizativa', 'Funcion', 'Area', 'Servicio', 'Sexo', 'TipoJornada']


In [18]:

numeric_transformer_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="mean"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor_tree = ColumnTransformer([
    ("num", numeric_transformer_tree, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor_linear = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocesadores preparados.")


Preprocesadores preparados.



## 6. Modelos de primera criba

Se plantea una comparación inicial de modelos de regresión equivalentes a las familias de algoritmos utilizadas como primera criba en trabajos de referencia, adaptadas al problema de predicción de días de IT.

Se incluyen:

- Ridge Regression como modelo lineal de referencia.
- Decision Tree Regressor.
- Random Forest Regressor.
- Extra Trees Regressor.
- Gradient Boosting Regressor.
- AdaBoost Regressor.
- XGBoost, si está disponible.
- CatBoost, si está disponible.
- LightGBM, si está disponible.

Los modelos se evaluarán sobre las cuatro salidas futuras.


In [19]:

from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

models = {
    "Ridge": Pipeline([
        ("preprocess", preprocessor_linear),
        ("model", Ridge(alpha=1.0))
    ]),

    "Decision Tree": Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", DecisionTreeRegressor(
            random_state=42,
            max_depth=12,
            min_samples_leaf=20
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1,
            max_depth=18,
            min_samples_leaf=10
        ))
    ]),

    "Extra Trees": Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", ExtraTreesRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1,
            max_depth=18,
            min_samples_leaf=10
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", GradientBoostingRegressor(
            random_state=42,
            n_estimators=100,
            max_depth=3,
            learning_rate=0.05
        ))
    ]),

    "AdaBoost": Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", AdaBoostRegressor(
            random_state=42,
            n_estimators=100,
            learning_rate=0.05
        ))
    ])
}

# XGBoost
try:
    from xgboost import XGBRegressor
    models["XGBoost"] = Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ])
    print("XGBoost disponible.")
except ImportError:
    print("XGBoost no está instalado. Se omitirá de la comparación.")

# CatBoost
try:
    from catboost import CatBoostRegressor
    print("CatBoost disponible, pero se incorporará después mediante su tratamiento nativo de categóricas.")
except ImportError:
    print("CatBoost no está instalado.")

# LightGBM
try:
    from lightgbm import LGBMRegressor
    models["LightGBM"] = Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        ))
    ])
    print("LightGBM disponible.")
except ImportError:
    print("LightGBM no está instalado.")

print("\nModelos configurados:")
for name in models:
    print("-", name)


XGBoost disponible.
CatBoost disponible, pero se incorporará después mediante su tratamiento nativo de categóricas.
LightGBM disponible.

Modelos configurados:
- Ridge
- Decision Tree
- Random Forest
- Extra Trees
- Gradient Boosting
- AdaBoost
- XGBoost
- LightGBM



## 7. Funciones de evaluación

Se calculan MAE, RMSE y R² de forma independiente para cada horizonte. 

**Nota:** dado que los objetivos presentan una elevada proporción de ceros, R² se interpretará conjuntamente con MAE y RMSE, y no como único criterio de selección.


In [20]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_predictions(y_true, y_pred, model_name, split_name):
    rows = []

    for i, target in enumerate(TARGETS):
        yt = y_true[target].to_numpy()
        yp = np.asarray(y_pred)[:, i]

        rows.append({
            "Modelo": model_name,
            "Conjunto": split_name,
            "Objetivo": target,
            "MAE": mean_absolute_error(yt, yp),
            "RMSE": np.sqrt(mean_squared_error(yt, yp)),
            "R2": r2_score(yt, yp)
        })

    return pd.DataFrame(rows)



## 8. Primera criba: entrenamiento y validación

> **Advertencia computacional:** el conjunto contiene más de 1,7 millones de registros. Los modelos de árboles/ensembles pueden requerir un tiempo y memoria considerablemente superiores al modelo lineal. Se recomienda ejecutar esta sección primero y observar el comportamiento del equipo antes de ampliar el número de estimadores.


In [21]:

results = []
fitted_models = {}

for name, model in models.items():
    print(f"\n{'='*70}")
    print(f"Entrenando: {name}")
    start = time.time()

    try:
        model.fit(X_train, y_train)
        elapsed = time.time() - start

        pred_val = model.predict(X_val)
        result = evaluate_predictions(y_val, pred_val, name, "Validation")
        result["Tiempo_entrenamiento_s"] = elapsed

        results.append(result)
        fitted_models[name] = model

        print(f"Completado en {elapsed/60:.2f} minutos.")
        display(result.round(4))

    except Exception as e:
        print(f"ERROR en {name}: {type(e).__name__}: {e}")



Entrenando: Ridge
Completado en 0.49 minutos.


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,Ridge,Validation,IT_Futura_W1,0.1748,0.6301,0.7056,29.1839
1,Ridge,Validation,IT_Futura_W2,0.2211,0.7162,0.6220,29.1839
2,Ridge,Validation,IT_Futura_W3,0.2580,0.7808,0.5532,29.1839
3,Ridge,Validation,IT_Futura_W4,0.2885,0.8301,0.4971,29.1839



Entrenando: Decision Tree
Completado en 0.89 minutos.


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,Decision Tree,Validation,IT_Futura_W1,0.1690,0.6259,0.7095,53.1275
1,Decision Tree,Validation,IT_Futura_W2,0.2154,0.7225,0.6154,53.1275
2,Decision Tree,Validation,IT_Futura_W3,0.2520,0.7941,0.5378,53.1275
3,Decision Tree,Validation,IT_Futura_W4,0.2819,0.8465,0.4770,53.1275



Entrenando: Random Forest
Completado en 25.55 minutos.


...


Entrenando: Extra Trees
Completado en 29.06 minutos.


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,Extra Trees,Validation,IT_Futura_W1,0.1693,0.6155,0.7191,1743.5998
1,Extra Trees,Validation,IT_Futura_W2,0.2154,0.7053,0.6335,1743.5998
2,Extra Trees,Validation,IT_Futura_W3,0.2521,0.7717,0.5636,1743.5998
3,Extra Trees,Validation,IT_Futura_W4,0.2821,0.8222,0.5066,1743.5998



Entrenando: Gradient Boosting
ERROR en Gradient Boosting: ValueError: y should be a 1d array, got an array of shape (1206215, 4) instead.

Entrenando: AdaBoost
ERROR en AdaBoost: ValueError: y should be a 1d array, got an array of shape (1206215, 4) instead.

Entrenando: XGBoost
Completado en 0.79 minutos.


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,XGBoost,Validation,IT_Futura_W1,0.1774,0.6147,0.7198,47.6045
1,XGBoost,Validation,IT_Futura_W2,0.2229,0.7024,0.6365,47.6045
2,XGBoost,Validation,IT_Futura_W3,0.2588,0.7685,0.5672,47.6045
3,XGBoost,Validation,IT_Futura_W4,0.2896,0.8179,0.5117,47.6045



Entrenando: LightGBM
ERROR en LightGBM: ValueError: y should be a 1d array, got an array of shape (1206215, 4) instead.


In [22]:
from sklearn.multioutput import MultiOutputRegressor

print("Configurando modelos multisalida...")

Configurando modelos multisalida...


In [23]:
models_retry = {}

models_retry["Gradient Boosting"] = Pipeline([
    ("preprocess", preprocessor_tree),
    ("model", MultiOutputRegressor(
        GradientBoostingRegressor(
            random_state=42,
            n_estimators=100,
            max_depth=3,
            learning_rate=0.05
        ),
        n_jobs=-1
    ))
])

models_retry["AdaBoost"] = Pipeline([
    ("preprocess", preprocessor_tree),
    ("model", MultiOutputRegressor(
        AdaBoostRegressor(
            random_state=42,
            n_estimators=100,
            learning_rate=0.05
        ),
        n_jobs=-1
    ))
])

print("Modelos preparados:")
for name in models_retry:
    print("-", name)

Modelos preparados:
- Gradient Boosting
- AdaBoost


In [24]:
for name, model in models_retry.items():

    print("\n" + "=" * 70)
    print(f"Entrenando: {name}")

    start = time.time()

    try:
        model.fit(X_train, y_train)

        elapsed = time.time() - start

        pred_val = model.predict(X_val)

        result = evaluate_predictions(
            y_val,
            pred_val,
            name,
            "Validation"
        )

        result["Tiempo_entrenamiento_s"] = elapsed

        results.append(result)
        fitted_models[name] = model

        print(f"Completado en {elapsed/60:.2f} minutos.")

        display(result.round(4))

    except Exception as e:
        print(f"ERROR en {name}: {type(e).__name__}: {e}")


Entrenando: Gradient Boosting
Completado en 12.96 minutos.


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,Gradient Boosting,Validation,IT_Futura_W1,0.1702,0.6139,0.7206,777.4636
1,Gradient Boosting,Validation,IT_Futura_W2,0.2162,0.7021,0.6368,777.4636
2,Gradient Boosting,Validation,IT_Futura_W3,0.2524,0.7676,0.5682,777.4636
3,Gradient Boosting,Validation,IT_Futura_W4,0.2827,0.8174,0.5123,777.4636



Entrenando: AdaBoost
Completado en 15.22 minutos.


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,AdaBoost,Validation,IT_Futura_W1,0.4454,0.7065,0.6299,912.9611
1,AdaBoost,Validation,IT_Futura_W2,0.5125,0.7934,0.5362,912.9611
2,AdaBoost,Validation,IT_Futura_W3,0.5684,0.8604,0.4575,912.9611
3,AdaBoost,Validation,IT_Futura_W4,0.6095,0.9086,0.3974,912.9611


In [25]:
models_retry["LightGBM"] = Pipeline([
    ("preprocess", preprocessor_tree),
    ("model", MultiOutputRegressor(
        LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        ),
        n_jobs=-1
    ))
])

In [26]:

if results:
    results_df = pd.concat(results, ignore_index=True)

    comparison = (
        results_df
        .groupby("Modelo")
        .agg(
            MAE_promedio=("MAE", "mean"),
            RMSE_promedio=("RMSE", "mean"),
            R2_promedio=("R2", "mean"),
            Tiempo_entrenamiento_s=("Tiempo_entrenamiento_s", "first")
        )
        .sort_values("MAE_promedio")
    )

    display(comparison.round(4))
else:
    print("No hay resultados disponibles.")


,MAE_promedio,RMSE_promedio,R2_promedio,Tiempo_entrenamiento_s
Modelo,,,,
Decision Tree,0.2296,0.7473,0.5849,53.1275
Extra Trees,0.2297,0.7287,0.6057,1743.5998
Gradient Boosting,0.2304,0.7252,0.6095,777.4636
Ridge,0.2356,0.7393,0.5945,29.1839
XGBoost,0.2372,0.7259,0.6088,47.6045
Random Forest,0.2400,0.7308,0.6033,1532.7522
AdaBoost,0.5339,0.8172,0.5052,912.9611


## 9. Comparación por horizonte

In [27]:

if results:
    tabla_mae = results_df.pivot(index="Modelo", columns="Objetivo", values="MAE")
    tabla_rmse = results_df.pivot(index="Modelo", columns="Objetivo", values="RMSE")
    tabla_r2 = results_df.pivot(index="Modelo", columns="Objetivo", values="R2")

    print("MAE por horizonte")
    display(tabla_mae.round(4))

    print("RMSE por horizonte")
    display(tabla_rmse.round(4))

    print("R² por horizonte")
    display(tabla_r2.round(4))


MAE por horizonte


Objetivo,IT_Futura_W1,IT_Futura_W2,IT_Futura_W3,IT_Futura_W4
Modelo,,,,
AdaBoost,0.4454,0.5125,0.5684,0.6095
Decision Tree,0.1690,0.2154,0.2520,0.2819
Extra Trees,0.1693,0.2154,0.2521,0.2821
Gradient Boosting,0.1702,0.2162,0.2524,0.2827
Random Forest,0.1761,0.2253,0.2638,0.2948
Ridge,0.1748,0.2211,0.2580,0.2885
XGBoost,0.1774,0.2229,0.2588,0.2896


RMSE por horizonte


Objetivo,IT_Futura_W1,IT_Futura_W2,IT_Futura_W3,IT_Futura_W4
Modelo,,,,
AdaBoost,0.7065,0.7934,0.8604,0.9086
Decision Tree,0.6259,0.7225,0.7941,0.8465
Extra Trees,0.6155,0.7053,0.7717,0.8222
Gradient Boosting,0.6139,0.7021,0.7676,0.8174
Random Forest,0.6167,0.7071,0.7742,0.8251
Ridge,0.6301,0.7162,0.7808,0.8301
XGBoost,0.6147,0.7024,0.7685,0.8179


R² por horizonte


Objetivo,IT_Futura_W1,IT_Futura_W2,IT_Futura_W3,IT_Futura_W4
Modelo,,,,
AdaBoost,0.6299,0.5362,0.4575,0.3974
Decision Tree,0.7095,0.6154,0.5378,0.4770
Extra Trees,0.7191,0.6335,0.5636,0.5066
Gradient Boosting,0.7206,0.6368,0.5682,0.5123
Random Forest,0.7180,0.6315,0.5608,0.5031
Ridge,0.7056,0.6220,0.5532,0.4971
XGBoost,0.7198,0.6365,0.5672,0.5117



## 10. Identificación de modelos candidatos

Esta sección realiza una primera identificación automática de los modelos con menor MAE promedio en Validation.

**No constituye todavía la selección definitiva del modelo**, ya que posteriormente deberán considerarse RMSE, R², comportamiento por horizonte, estabilidad temporal y coste computacional.


In [28]:

if results:
    candidatos = comparison.head(4).copy()
    display(candidatos.round(4))
    print("Modelos candidatos para la fase de optimización:")
    for model_name in candidatos.index:
        print("-", model_name)


,MAE_promedio,RMSE_promedio,R2_promedio,Tiempo_entrenamiento_s
Modelo,,,,
Decision Tree,0.2296,0.7473,0.5849,53.1275
Extra Trees,0.2297,0.7287,0.6057,1743.5998
Gradient Boosting,0.2304,0.7252,0.6095,777.4636
Ridge,0.2356,0.7393,0.5945,29.1839


Modelos candidatos para la fase de optimización:
- Decision Tree
- Extra Trees
- Gradient Boosting
- Ridge



## 11. Evaluación preliminar sobre Test

Esta celda se deja preparada pero **no debe utilizarse para seleccionar modelos**.

El conjunto Test debe permanecer aislado hasta que se haya definido el modelo final y sus hiperparámetros. Una vez completada la optimización, se podrá ejecutar esta evaluación como comprobación final del rendimiento fuera de muestra.


In [29]:

# NO EJECUTAR PARA LA SELECCIÓN DEL MODELO.
# Se habilitará una vez finalizada la optimización.

EVALUAR_TEST = False

if EVALUAR_TEST:
    final_test_results = []

    for name, model in fitted_models.items():
        pred_test = model.predict(X_test)
        r = evaluate_predictions(y_test, pred_test, name, "Test")
        r["Tiempo_entrenamiento_s"] = np.nan
        final_test_results.append(r)

    test_results_df = pd.concat(final_test_results, ignore_index=True)
    display(test_results_df.round(4))
else:
    print("Test permanece aislado para la fase final.")


Test permanece aislado para la fase final.



## 12. Exportación de resultados de la primera criba

Los resultados se guardan para poder documentar la comparación de modelos y utilizarlos posteriormente en la fase de optimización.


In [30]:

OUTPUT_RESULTS = BASE_DIR / "Resultados_Primera_Criba_Modelos_IT.csv"

if results:
    results_df.to_csv(
        OUTPUT_RESULTS,
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )
    print(f"Resultados guardados en: {OUTPUT_RESULTS}")
else:
    print("No se guardaron resultados porque no se completó ningún modelo.")


Resultados guardados en: c:\Users\Oscar\OneDrive\Documents\TFM\10 Modelado\Resultados_Primera_Criba_Modelos_IT.csv


Punto 13


In [49]:
# ============================================================
# 13. MODELOS ADICIONALES
# 13.1 Preparación y evaluación
# ============================================================

import os
import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.multioutput import MultiOutputRegressor

print("=" * 70)
print("MODELOS ADICIONALES")
print("=" * 70)

# ------------------------------------------------------------
# Objetivos de predicción
# ------------------------------------------------------------

target_names_adicionales = [
    "IT_Futura_W1",
    "IT_Futura_W2",
    "IT_Futura_W3",
    "IT_Futura_W4"
]

# ------------------------------------------------------------
# Lista independiente de resultados
# ------------------------------------------------------------

results_adicionales = []

# ------------------------------------------------------------
# Función de evaluación
# ------------------------------------------------------------

def evaluar_modelo_adicional(
    y_real,
    y_pred,
    target_names
):
    """
    Calcula MAE, RMSE y R² para cada horizonte
    y obtiene el promedio de las cuatro métricas.
    """

    resultados_horizonte = []

    for i, target in enumerate(target_names):

        y_real_h = np.asarray(y_real.iloc[:, i])
        y_pred_h = np.asarray(y_pred[:, i])

        mae_h = mean_absolute_error(
            y_real_h,
            y_pred_h
        )

        rmse_h = np.sqrt(
            mean_squared_error(
                y_real_h,
                y_pred_h
            )
        )

        r2_h = r2_score(
            y_real_h,
            y_pred_h
        )

        resultados_horizonte.append({
            "Modelo": None,
            "Conjunto": "Validation",
            "Objetivo": target,
            "MAE": mae_h,
            "RMSE": rmse_h,
            "R2": r2_h,
            "Tiempo_entrenamiento_s": None
        })

    detalle = pd.DataFrame(
        resultados_horizonte
    )

    # Promedios de las métricas de los cuatro horizontes
    mae_promedio = detalle["MAE"].mean()
    rmse_promedio = detalle["RMSE"].mean()
    r2_promedio = detalle["R2"].mean()

    return (
        detalle,
        mae_promedio,
        rmse_promedio,
        r2_promedio
    )


print("\nConfiguración de modelos adicionales preparada.")
print(f"Objetivos: {target_names_adicionales}")

MODELOS ADICIONALES

Configuración de modelos adicionales preparada.
Objetivos: ['IT_Futura_W1', 'IT_Futura_W2', 'IT_Futura_W3', 'IT_Futura_W4']


In [50]:
# ============================================================
# 13.2 LIGHTGBM
# ============================================================

from lightgbm import LGBMRegressor

print("=" * 70)
print("ENTRENANDO LIGHTGBM")
print("=" * 70)

# ------------------------------------------------------------
# Modelo
# ------------------------------------------------------------

lightgbm_model = Pipeline([
    (
        "preprocess",
        preprocessor_tree
    ),
    (
        "model",
        MultiOutputRegressor(
            LGBMRegressor(
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                max_depth=-1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1,
                verbosity=-1
            ),
            n_jobs=-1
        )
    )
])

# ------------------------------------------------------------
# Entrenamiento
# ------------------------------------------------------------

inicio_lgbm = time.time()

lightgbm_model.fit(
    X_train,
    y_train
)

tiempo_lgbm = time.time() - inicio_lgbm

print(
    f"\nLightGBM entrenado en "
    f"{tiempo_lgbm / 60:.2f} minutos."
)

# ------------------------------------------------------------
# Predicción
# ------------------------------------------------------------

y_pred_lgbm = lightgbm_model.predict(
    X_val
)

# ------------------------------------------------------------
# Evaluación
# ------------------------------------------------------------

detalle_lgbm, mae_lgbm, rmse_lgbm, r2_lgbm = (
    evaluar_modelo_adicional(
        y_val,
        y_pred_lgbm,
        target_names_adicionales
    )
)

# Añadir información del modelo
detalle_lgbm["Modelo"] = "LightGBM"
detalle_lgbm["Tiempo_entrenamiento_s"] = tiempo_lgbm

# ------------------------------------------------------------
# Mostrar resultados por horizonte
# ------------------------------------------------------------

print("\nRESULTADOS LIGHTGBM POR HORIZONTE")
print("-" * 70)

display(
    detalle_lgbm[
        [
            "Modelo",
            "Conjunto",
            "Objetivo",
            "MAE",
            "RMSE",
            "R2",
            "Tiempo_entrenamiento_s"
        ]
    ]
)

# ------------------------------------------------------------
# Mostrar promedio
# ------------------------------------------------------------

print("\nPROMEDIO LIGHTGBM")
print("-" * 70)

print(f"MAE promedio:  {mae_lgbm:.4f}")
print(f"RMSE promedio: {rmse_lgbm:.4f}")
print(f"R² promedio:   {r2_lgbm:.4f}")
print(f"Tiempo:        {tiempo_lgbm / 60:.2f} minutos")

# ------------------------------------------------------------
# Guardar resultado resumen
# ------------------------------------------------------------

results_adicionales.append({
    "Modelo": "LightGBM",
    "MAE_Promedio": mae_lgbm,
    "RMSE_Promedio": rmse_lgbm,
    "R2_Promedio": r2_lgbm,
    "Tiempo_entrenamiento_s": tiempo_lgbm
})

# ------------------------------------------------------------
# Guardar detalle
# ------------------------------------------------------------

detalle_adicionales = detalle_lgbm.copy()

print("\n✓ LightGBM añadido correctamente a los resultados.")

ENTRENANDO LIGHTGBM

LightGBM entrenado en 0.43 minutos.

RESULTADOS LIGHTGBM POR HORIZONTE
----------------------------------------------------------------------


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,LightGBM,Validation,IT_Futura_W1,0.174129,0.613834,0.720617,25.534973
1,LightGBM,Validation,IT_Futura_W2,0.220487,0.702504,0.636365,25.534973
2,LightGBM,Validation,IT_Futura_W3,0.255115,0.767979,0.567764,25.534973
3,LightGBM,Validation,IT_Futura_W4,0.285581,0.817936,0.511686,25.534973



PROMEDIO LIGHTGBM
----------------------------------------------------------------------
MAE promedio:  0.2338
RMSE promedio: 0.7256
R² promedio:   0.6091
Tiempo:        0.43 minutos

✓ LightGBM añadido correctamente a los resultados.


In [51]:
# ============================================================
# 13.3 CATBOOST
# ============================================================

from catboost import CatBoostRegressor

print("=" * 70)
print("ENTRENANDO CATBOOST")
print("=" * 70)

# ------------------------------------------------------------
# Copias independientes de los datos
# ------------------------------------------------------------

X_train_cb = X_train.copy()
X_val_cb = X_val.copy()

# ------------------------------------------------------------
# Identificar posiciones de variables categóricas
# ------------------------------------------------------------

cat_features_idx = [
    X_train_cb.columns.get_loc(col)
    for col in categorical_features
]

# ------------------------------------------------------------
# Preparar variables categóricas
# ------------------------------------------------------------

for col in categorical_features:

    X_train_cb[col] = (
        X_train_cb[col]
        .fillna("Otros")
        .astype(str)
    )

    X_val_cb[col] = (
        X_val_cb[col]
        .fillna("Otros")
        .astype(str)
    )

print(
    f"\nVariables categóricas utilizadas: "
    f"{len(categorical_features)}"
)

print(
    f"Variables numéricas utilizadas: "
    f"{len(numeric_features)}"
)

# ------------------------------------------------------------
# Configuración inicial GPU
# ------------------------------------------------------------

catboost_model = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiRMSE",
    eval_metric="MultiRMSE",
    random_seed=42,
    verbose=100,
    task_type="GPU",
    devices="0",
    allow_writing_files=False
)

# ------------------------------------------------------------
# Entrenamiento
# ------------------------------------------------------------

inicio_catboost = time.time()

try:

    print("\nIntentando ejecutar CatBoost utilizando GPU...")

    catboost_model.fit(
        X_train_cb,
        y_train,
        cat_features=cat_features_idx,
        eval_set=(X_val_cb, y_val),
        early_stopping_rounds=50
    )

    modo_catboost = "GPU"

except Exception as error_gpu:

    print("\n⚠️ No fue posible ejecutar CatBoost en GPU.")
    print("Se realizará un segundo intento utilizando CPU.")
    print(f"Detalle del error GPU: {error_gpu}")

    catboost_model = CatBoostRegressor(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiRMSE",
        eval_metric="MultiRMSE",
        random_seed=42,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    catboost_model.fit(
        X_train_cb,
        y_train,
        cat_features=cat_features_idx,
        eval_set=(X_val_cb, y_val),
        early_stopping_rounds=50
    )

    modo_catboost = "CPU"

tiempo_catboost = time.time() - inicio_catboost

print(
    f"\nCatBoost entrenado en "
    f"{tiempo_catboost / 60:.2f} minutos."
)

print(
    f"Modo de ejecución: {modo_catboost}"
)

# ------------------------------------------------------------
# Predicción
# ------------------------------------------------------------

y_pred_catboost = catboost_model.predict(
    X_val_cb
)

# ------------------------------------------------------------
# Asegurar formato matricial
# ------------------------------------------------------------

y_pred_catboost = np.asarray(
    y_pred_catboost
)

# ------------------------------------------------------------
# Evaluación
# ------------------------------------------------------------

detalle_catboost, mae_catboost, rmse_catboost, r2_catboost = (
    evaluar_modelo_adicional(
        y_val,
        y_pred_catboost,
        target_names_adicionales
    )
)

# Añadir información del modelo
detalle_catboost["Modelo"] = "CatBoost"
detalle_catboost["Tiempo_entrenamiento_s"] = tiempo_catboost

# ------------------------------------------------------------
# Mostrar resultados por horizonte
# ------------------------------------------------------------

print("\nRESULTADOS CATBOOST POR HORIZONTE")
print("-" * 70)

display(
    detalle_catboost[
        [
            "Modelo",
            "Conjunto",
            "Objetivo",
            "MAE",
            "RMSE",
            "R2",
            "Tiempo_entrenamiento_s"
        ]
    ]
)

# ------------------------------------------------------------
# Mostrar promedio
# ------------------------------------------------------------

print("\nPROMEDIO CATBOOST")
print("-" * 70)

print(f"MAE promedio:  {mae_catboost:.4f}")
print(f"RMSE promedio: {rmse_catboost:.4f}")
print(f"R² promedio:   {r2_catboost:.4f}")
print(f"Tiempo:        {tiempo_catboost / 60:.2f} minutos")

# ------------------------------------------------------------
# Guardar resultado resumen
# ------------------------------------------------------------

results_adicionales.append({
    "Modelo": "CatBoost",
    "MAE_Promedio": mae_catboost,
    "RMSE_Promedio": rmse_catboost,
    "R2_Promedio": r2_catboost,
    "Tiempo_entrenamiento_s": tiempo_catboost
})

# ------------------------------------------------------------
# Añadir al detalle
# ------------------------------------------------------------

detalle_adicionales = pd.concat(
    [
        detalle_adicionales,
        detalle_catboost
    ],
    ignore_index=True
)

print("\n✓ CatBoost añadido correctamente a los resultados.")

ENTRENANDO CATBOOST

Variables categóricas utilizadas: 9
Variables numéricas utilizadas: 27

Intentando ejecutar CatBoost utilizando GPU...
0:	learn: 2.1639534	test: 2.2640304	best: 2.2640304 (0)	total: 45.1ms	remaining: 22.5s
100:	learn: 1.3749819	test: 1.4565939	best: 1.4565939 (100)	total: 4.09s	remaining: 16.2s
bestTest = 1.456374545
bestIteration = 122
Shrink model to first 123 iterations.

CatBoost entrenado en 0.24 minutos.
Modo de ejecución: GPU

RESULTADOS CATBOOST POR HORIZONTE
----------------------------------------------------------------------


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,CatBoost,Validation,IT_Futura_W1,0.165627,0.613190,0.721203,14.472439
1,CatBoost,Validation,IT_Futura_W2,0.210768,0.701121,0.637796,14.472439
2,CatBoost,Validation,IT_Futura_W3,0.246625,0.766439,0.569497,14.472439
3,CatBoost,Validation,IT_Futura_W4,0.276703,0.816104,0.513872,14.472439



PROMEDIO CATBOOST
----------------------------------------------------------------------
MAE promedio:  0.2249
RMSE promedio: 0.7242
R² promedio:   0.6106
Tiempo:        0.24 minutos

✓ CatBoost añadido correctamente a los resultados.


In [52]:
# ============================================================
# 13.3 CATBOOST
# ============================================================

from catboost import CatBoostRegressor

print("=" * 70)
print("ENTRENANDO CATBOOST")
print("=" * 70)

# ------------------------------------------------------------
# Copias independientes de los datos
# ------------------------------------------------------------

X_train_cb = X_train.copy()
X_val_cb = X_val.copy()

# ------------------------------------------------------------
# Identificar posiciones de variables categóricas
# ------------------------------------------------------------

cat_features_idx = [
    X_train_cb.columns.get_loc(col)
    for col in categorical_features
]

# ------------------------------------------------------------
# Preparar variables categóricas
# ------------------------------------------------------------

for col in categorical_features:

    X_train_cb[col] = (
        X_train_cb[col]
        .fillna("Otros")
        .astype(str)
    )

    X_val_cb[col] = (
        X_val_cb[col]
        .fillna("Otros")
        .astype(str)
    )

print(
    f"\nVariables categóricas utilizadas: "
    f"{len(categorical_features)}"
)

print(
    f"Variables numéricas utilizadas: "
    f"{len(numeric_features)}"
)

# ------------------------------------------------------------
# Configuración inicial GPU
# ------------------------------------------------------------

catboost_model = CatBoostRegressor(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiRMSE",
    eval_metric="MultiRMSE",
    random_seed=42,
    verbose=100,
    task_type="GPU",
    devices="0",
    allow_writing_files=False
)

# ------------------------------------------------------------
# Entrenamiento
# ------------------------------------------------------------

inicio_catboost = time.time()

try:

    print("\nIntentando ejecutar CatBoost utilizando GPU...")

    catboost_model.fit(
        X_train_cb,
        y_train,
        cat_features=cat_features_idx,
        eval_set=(X_val_cb, y_val),
        early_stopping_rounds=50
    )

    modo_catboost = "GPU"

except Exception as error_gpu:

    print("\n⚠️ No fue posible ejecutar CatBoost en GPU.")
    print("Se realizará un segundo intento utilizando CPU.")
    print(f"Detalle del error GPU: {error_gpu}")

    catboost_model = CatBoostRegressor(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiRMSE",
        eval_metric="MultiRMSE",
        random_seed=42,
        verbose=100,
        thread_count=-1,
        allow_writing_files=False
    )

    catboost_model.fit(
        X_train_cb,
        y_train,
        cat_features=cat_features_idx,
        eval_set=(X_val_cb, y_val),
        early_stopping_rounds=50
    )

    modo_catboost = "CPU"

tiempo_catboost = time.time() - inicio_catboost

print(
    f"\nCatBoost entrenado en "
    f"{tiempo_catboost / 60:.2f} minutos."
)

print(
    f"Modo de ejecución: {modo_catboost}"
)

# ------------------------------------------------------------
# Predicción
# ------------------------------------------------------------

y_pred_catboost = catboost_model.predict(
    X_val_cb
)

# ------------------------------------------------------------
# Asegurar formato matricial
# ------------------------------------------------------------

y_pred_catboost = np.asarray(
    y_pred_catboost
)

# ------------------------------------------------------------
# Evaluación
# ------------------------------------------------------------

detalle_catboost, mae_catboost, rmse_catboost, r2_catboost = (
    evaluar_modelo_adicional(
        y_val,
        y_pred_catboost,
        target_names_adicionales
    )
)

# Añadir información del modelo
detalle_catboost["Modelo"] = "CatBoost"
detalle_catboost["Tiempo_entrenamiento_s"] = tiempo_catboost

# ------------------------------------------------------------
# Mostrar resultados por horizonte
# ------------------------------------------------------------

print("\nRESULTADOS CATBOOST POR HORIZONTE")
print("-" * 70)

display(
    detalle_catboost[
        [
            "Modelo",
            "Conjunto",
            "Objetivo",
            "MAE",
            "RMSE",
            "R2",
            "Tiempo_entrenamiento_s"
        ]
    ]
)

# ------------------------------------------------------------
# Mostrar promedio
# ------------------------------------------------------------

print("\nPROMEDIO CATBOOST")
print("-" * 70)

print(f"MAE promedio:  {mae_catboost:.4f}")
print(f"RMSE promedio: {rmse_catboost:.4f}")
print(f"R² promedio:   {r2_catboost:.4f}")
print(f"Tiempo:        {tiempo_catboost / 60:.2f} minutos")

# ------------------------------------------------------------
# Guardar resultado resumen
# ------------------------------------------------------------

results_adicionales.append({
    "Modelo": "CatBoost",
    "MAE_Promedio": mae_catboost,
    "RMSE_Promedio": rmse_catboost,
    "R2_Promedio": r2_catboost,
    "Tiempo_entrenamiento_s": tiempo_catboost
})

# ------------------------------------------------------------
# Añadir al detalle
# ------------------------------------------------------------

detalle_adicionales = pd.concat(
    [
        detalle_adicionales,
        detalle_catboost
    ],
    ignore_index=True
)

print("\n✓ CatBoost añadido correctamente a los resultados.")

ENTRENANDO CATBOOST

Variables categóricas utilizadas: 9
Variables numéricas utilizadas: 27

Intentando ejecutar CatBoost utilizando GPU...
0:	learn: 2.1639535	test: 2.2640301	best: 2.2640301 (0)	total: 45.3ms	remaining: 22.6s
100:	learn: 1.3749818	test: 1.4565936	best: 1.4565936 (100)	total: 4.13s	remaining: 16.3s
bestTest = 1.456374231
bestIteration = 122
Shrink model to first 123 iterations.

CatBoost entrenado en 0.25 minutos.
Modo de ejecución: GPU

RESULTADOS CATBOOST POR HORIZONTE
----------------------------------------------------------------------


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,CatBoost,Validation,IT_Futura_W1,0.165627,0.613190,0.721203,14.716453
1,CatBoost,Validation,IT_Futura_W2,0.210768,0.701121,0.637796,14.716453
2,CatBoost,Validation,IT_Futura_W3,0.246625,0.766439,0.569497,14.716453
3,CatBoost,Validation,IT_Futura_W4,0.276703,0.816104,0.513872,14.716453



PROMEDIO CATBOOST
----------------------------------------------------------------------
MAE promedio:  0.2249
RMSE promedio: 0.7242
R² promedio:   0.6106
Tiempo:        0.25 minutos

✓ CatBoost añadido correctamente a los resultados.


In [53]:
# ============================================================
# 13.5 COMPARACIÓN POR HORIZONTE
# ============================================================

print("=" * 70)
print("RESULTADOS DETALLADOS POR HORIZONTE")
print("=" * 70)

detalle_adicionales_ordenado = (
    detalle_adicionales
    .sort_values(
        by=["Objetivo", "R2"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(
    detalle_adicionales_ordenado
)

RESULTADOS DETALLADOS POR HORIZONTE


,Modelo,Conjunto,Objetivo,MAE,RMSE,R2,Tiempo_entrenamiento_s
0,CatBoost,Validation,IT_Futura_W1,0.165627,0.613190,0.721203,14.472439
1,CatBoost,Validation,IT_Futura_W1,0.165627,0.613190,0.721203,14.716453
2,LightGBM,Validation,IT_Futura_W1,0.174129,0.613834,0.720617,25.534973
3,CatBoost,Validation,IT_Futura_W2,0.210768,0.701121,0.637796,14.472439
4,CatBoost,Validation,IT_Futura_W2,0.210768,0.701121,0.637796,14.716453
5,LightGBM,Validation,IT_Futura_W2,0.220487,0.702504,0.636365,25.534973
6,CatBoost,Validation,IT_Futura_W3,0.246625,0.766439,0.569497,14.472439
7,CatBoost,Validation,IT_Futura_W3,0.246625,0.766439,0.569497,14.716453
8,LightGBM,Validation,IT_Futura_W3,0.255115,0.767979,0.567764,25.534973
9,CatBoost,Validation,IT_Futura_W4,0.276703,0.816104,0.513872,14.472439


In [55]:
# ============================================================
# 13.6 EXPORTACIÓN DE RESULTADOS ADICIONALES
# ============================================================

print("=" * 70)
print("EXPORTACIÓN DE RESULTADOS ADICIONALES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Comprobar que existen resultados de los modelos
# ------------------------------------------------------------

if "results_adicionales" not in globals():
    raise NameError(
        "No existe 'results_adicionales'. "
        "Los resultados de LightGBM/CatBoost no están disponibles."
    )

if len(results_adicionales) == 0:
    raise ValueError(
        "La lista 'results_adicionales' está vacía. "
        "No hay resultados para exportar."
    )

# ------------------------------------------------------------
# 2. Crear el DataFrame de resumen AQUÍ
#    No dependemos de la celda 13.4
# ------------------------------------------------------------

resultados_adicionales_df = pd.DataFrame(
    results_adicionales
)

# Ordenar por R² promedio
resultados_adicionales_df = (
    resultados_adicionales_df
    .sort_values(
        by="R2_Promedio",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Crear carpeta de resultados
# ------------------------------------------------------------

carpeta_resultados = os.path.join(
    os.getcwd(),
    "Resultados"
)

os.makedirs(
    carpeta_resultados,
    exist_ok=True
)

# ------------------------------------------------------------
# 4. Definir rutas
# ------------------------------------------------------------

ruta_resumen = os.path.join(
    carpeta_resultados,
    "Resultados_Modelos_Adicionales_IT.csv"
)

ruta_detalle = os.path.join(
    carpeta_resultados,
    "Resultados_Modelos_Adicionales_IT_Detalle.csv"
)

# ------------------------------------------------------------
# 5. Exportar resumen
# ------------------------------------------------------------

resultados_adicionales_df.to_csv(
    ruta_resumen,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 6. Recuperar detalle por horizonte
# ------------------------------------------------------------

# Si ya existe el detalle consolidado, se utiliza directamente.
if "detalle_adicionales_ordenado" in globals():

    detalle_final = detalle_adicionales_ordenado.copy()

# Si no existe, intentamos reconstruirlo a partir de
# los detalles individuales de los modelos.
else:

    detalles_disponibles = []

    if "detalle_lgbm" in globals():
        detalles_disponibles.append(
            detalle_lgbm.copy()
        )

    if "detalle_catboost" in globals():
        detalles_disponibles.append(
            detalle_catboost.copy()
        )

    if len(detalles_disponibles) > 0:

        detalle_final = pd.concat(
            detalles_disponibles,
            ignore_index=True
        )

        detalle_final = (
            detalle_final
            .sort_values(
                by=["Objetivo", "R2"],
                ascending=[True, False]
            )
            .reset_index(drop=True)
        )

    else:

        detalle_final = pd.DataFrame()

# ------------------------------------------------------------
# 7. Exportar detalle si existe
# ------------------------------------------------------------

if not detalle_final.empty:

    detalle_final.to_csv(
        ruta_detalle,
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    detalle_exportado = True

else:

    detalle_exportado = False

# ------------------------------------------------------------
# 8. Confirmación
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPORTACIÓN COMPLETADA")
print("=" * 70)

print("\nArchivo resumen:")
print(ruta_resumen)

print(
    f"\nModelos exportados: "
    f"{len(resultados_adicionales_df)}"
)

print("\nModelos incluidos:")

for modelo in resultados_adicionales_df["Modelo"]:
    print(f"  - {modelo}")

if detalle_exportado:

    print("\nArchivo detalle por horizonte:")
    print(ruta_detalle)

else:

    print(
        "\n⚠️ No se encontró información de detalle "
        "por horizonte para exportar."
    )

print("\nResumen:")
display(resultados_adicionales_df)

EXPORTACIÓN DE RESULTADOS ADICIONALES

EXPORTACIÓN COMPLETADA

Archivo resumen:
c:\Users\Oscar\OneDrive\Documents\TFM\10 Modelado\Resultados\Resultados_Modelos_Adicionales_IT.csv

Modelos exportados: 3

Modelos incluidos:
  - CatBoost
  - CatBoost
  - LightGBM

Archivo detalle por horizonte:
c:\Users\Oscar\OneDrive\Documents\TFM\10 Modelado\Resultados\Resultados_Modelos_Adicionales_IT_Detalle.csv

Resumen:


,Modelo,MAE_Promedio,RMSE_Promedio,R2_Promedio,Tiempo_entrenamiento_s
0,CatBoost,0.224931,0.724213,0.610592,14.472439
1,CatBoost,0.224931,0.724213,0.610592,14.716453
2,LightGBM,0.233828,0.725563,0.609108,25.534973



# Próxima fase

Una vez completada esta primera criba, el siguiente paso será:

1. analizar los resultados por horizonte;
2. seleccionar los modelos candidatos;
3. optimizar sus hiperparámetros;
4. volver a comparar su rendimiento;
5. seleccionar el modelo final;
6. realizar una única evaluación final sobre Test;
7. interpretar las variables más relevantes del modelo seleccionado.

**No se debe realizar todavía una selección definitiva de variables predictoras:** las 36 variables preparadas constituyen el conjunto inicial de predictores y su utilidad se evaluará posteriormente mediante los modelos.
